In [1]:
import sys

sys.path.append("..")

In [2]:
from src.utils.config import (load_all_configs)
from src.utils.reproducibility import (set_global_seed)
from src.data.loaders import (load_product_dataset)
from src.embeddings.generator import (EmbeddingGenerator)
from src.embeddings.storage import (save_embeddings)
from src.experiments.dimension_sweep import (DimensionSweepExperiment)
from src.visualization.metric_plots import (MetricPlotter)
from src.utils.paths import (experiment_path, project_path)

In [3]:
config = load_all_configs(
    "dimension_sweep"
)
config

{'dataset': {'path': 'data/raw/some_product_names.xlsx',
  'text_column': 'Name'},
 'embeddings': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
  'batch_size': 32,
  'normalize': True},
 'experiment': {'random_state': 42, 'name': 'dimension_sweep'},
 'storage': {'embeddings_path': 'data/embeddings/product_embeddings.npy'},
 'umap': {'n_neighbors': 15,
  'min_dist': 0.1,
  'metric': 'cosine',
  'random_state': 42},
 'dimension_sweep': {'dimensions': [256, 128, 64, 32, 16, 8, 4, 2]},
 'kmeans': {'enabled': True, 'parameters': {'n_clusters': [5, 10, 15]}},
 'dbscan': {'enabled': True,
  'parameters': {'eps': [0.3, 0.5, 0.7], 'min_samples': [5, 10]}},
 'hdbscan': {'enabled': True,
  'parameters': {'min_cluster_size': [10, 20, 50]}},
 'agglomerative': {'enabled': True,
  'parameters': {'n_clusters': [5, 10, 15], 'linkage': ['ward', 'average']}},
 'outputs': {'results_dir': 'results',
  'figures_dir': 'figures',
  'logs_dir': 'logs'},
 'dimensions': [256, 128, 64, 32, 16, 8, 4, 2]

In [4]:
set_global_seed(config["experiment"]["random_state"])

In [5]:
df = load_product_dataset(
    dataset_path=config["dataset"]["path"],
    text_column=config["dataset"]["text_column"]
)

df.head()

,id,Name,processed_text
0,1,Лампа комактна люмінісцентна 2G7 11W/840 DULUX...,лампа комактна люмінісцентна 2g7 11w 840 dulux...
1,2,Клемна колодка в корпусі Аско ТВ 1504 A0130050002,клемна колодка в корпусі аско тв 1504 a0130050002
2,3,Лампа TV Osram Е14 25W T-25 синя,лампа tv osram е14 25w t 25 синя
3,4,Лампа для холодильника РП Іскра Е14 230B 15W S...,лампа для холодильника рп іскра е14 230b 15w s...
4,5,Лампа для духовки Osram SPC.T OVEN CL 15W 230V...,лампа для духовки osram spc t oven cl 15w 230v...


In [6]:
generator = EmbeddingGenerator(
    model_name=config["embeddings"]["model_name"]
)

embeddings = generator.generate(
    df["processed_text"].tolist(),
    batch_size=config["embeddings"]["batch_size"],
    normalize_embeddings=config["embeddings"]["normalize"]
)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)

In [8]:
save_embeddings(
    embeddings,
    config["storage"]["embeddings_path"]
)

In [9]:
experiment = (
    DimensionSweepExperiment(
        config
    )
)

In [10]:
results_df = experiment.run(embeddings)

C:\Users\Acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by set

In [11]:
results_df.head(20)

,algorithm,dimension,n_clusters,silhouette,davies_bouldin,calinski_harabasz,cluster_count,noise_ratio,trustworthiness,knn_preservation,eps,min_samples,min_cluster_size,linkage
0,kmeans,256,5.0,0.357396,0.801960,231.257767,5,0.000,0.938834,0.604667,NaN,NaN,NaN,NaN
1,kmeans,256,10.0,0.370186,0.794479,337.094543,10,0.000,0.938834,0.604667,NaN,NaN,NaN,NaN
2,kmeans,256,15.0,0.418112,0.588700,465.004822,15,0.000,0.938834,0.604667,NaN,NaN,NaN,NaN
3,dbscan,256,NaN,0.638460,1.014467,418.419250,77,0.060,0.938834,0.604667,0.3,5.0,NaN,NaN
4,dbscan,256,NaN,0.160289,1.023613,50.726112,33,0.452,0.938834,0.604667,0.3,10.0,NaN,NaN
5,dbscan,256,NaN,0.678290,0.841377,1297.744507,55,0.022,0.938834,0.604667,0.5,5.0,NaN,NaN
6,dbscan,256,NaN,0.487587,1.440991,71.499008,39,0.184,0.938834,0.604667,0.5,10.0,NaN,NaN
7,dbscan,256,NaN,0.638518,0.555308,2042.327148,48,0.007,0.938834,0.604667,0.7,5.0,NaN,NaN
8,dbscan,256,NaN,0.511905,1.436182,104.454811,32,0.138,0.938834,0.604667,0.7,10.0,NaN,NaN
9,hdbscan,256,NaN,0.536413,1.447007,102.565712,33,0.149,0.938834,0.604667,NaN,NaN,10.0,NaN


In [12]:
results_df.describe()

,dimension,n_clusters,silhouette,davies_bouldin,calinski_harabasz,cluster_count,noise_ratio,trustworthiness,knn_preservation,eps,min_samples,min_cluster_size
count,144.000000,72.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,48.000000,48.000000,24.000000
mean,63.750000,10.000000,0.439289,1.076483,456.149048,23.756944,0.088340,0.939983,0.609483,0.500000,7.500000,26.666667
std,83.104745,4.111132,0.132295,0.641022,591.895935,20.477600,0.134655,0.004458,0.002779,0.165027,2.526456,17.362295
min,2.000000,5.000000,0.001916,0.322574,38.824757,2.000000,0.000000,0.932724,0.604667,0.300000,5.000000,10.000000
25%,7.000000,5.000000,0.369865,0.640671,105.649008,10.000000,0.000000,0.938187,0.607250,0.300000,5.000000,10.000000
50%,24.000000,10.000000,0.452444,0.826283,271.652161,15.000000,0.002000,0.938827,0.610067,0.500000,7.500000,20.000000
75%,80.000000,15.000000,0.510971,1.405979,457.074295,36.250000,0.156750,0.941657,0.611833,0.700000,10.000000,50.000000
max,256.000000,15.000000,0.693245,3.792517,3350.724121,80.000000,0.576000,0.948896,0.612733,0.700000,10.000000,50.000000


In [13]:
experiment_name = (
    config["experiment"]["name"]
)

results_path = experiment_path(
    experiment_name,
    "results",
    "dimension_sweep_results.csv"
)

figures_dir = experiment_path(
    experiment_name,
    "figures"
)

In [14]:
results_df.to_csv(
    results_path,
    index=False
)

In [15]:
plotter = MetricPlotter(
    figures_dir=figures_dir
)

In [16]:
algorithms = results_df[
    "algorithm"
].unique()

In [17]:
metrics = [
    "silhouette",
    "davies_bouldin",
    "calinski_harabasz"
]

In [18]:
for algorithm in algorithms:
    for metric in metrics:
        plotter.plot_metric_by_dimension(
            results_df,
            metric,
            algorithm
        )

In [19]:
plotter.plot_preservation_metrics(
    results_df
)